<div dir="rtl" align="right">

# المُرشِّحُ الوسيطُ \(Median Filter\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يَستبدلُ المُرشِّحُ الوسيطُ كلَّ عيّنةٍ بقيمةِ الوسيطِ بين جيرانِها. يَتميّزُ بمقاومةِ الشوائبِ (القيمِ الشاذّةِ) لأنّه لا يتأثّرُ بالقيمِ المتطرفةِ. نَحقنُ شوكةً اصطناعيةً في العيّنةِ 1000 لنُلاحظَ هذه المقاومةَ.

## المُخرجاتُ المُتوقّعةُ

- الشوكةُ الاصطناعيةُ تَظهرُ بوضوحٍ في الإشارةِ الأصليةِ
- المُرشِّحُ الوسيطُ يُزيلُ الشوكةَ تمامًا حتى مع النوافذِ الصغيرةِ
- النوافذُ الأكبرُ تُنعمُ الإشارةَ أكثرَ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| معدّلُ الأخذِ | 200 Hz | عيّنةٌ كلَّ 5 ms |
| النوافذُ | 5, 11, 21 | أحجامُ النافذةِ |
| موضعُ الشوكةِ | 1000 | العيّنةُ 1000 |
| قيمةُ الشوكةِ | +200 uV | زيادةٌ اصطناعيةٌ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ المُرشِّحِ

نَحقنُ شوكةً اصطناعيةً (+200 uV) في العيّنةِ 1000، ثمّ نَستخدمُ `scipy.signal.medfilt` مع ثلاثةِ أحجامِ نوافذَ. الوسيطُ يَتجاهلُ القيمَ المتطرفةَ فعليًا.

</div>

In [ ]:
from scipy.signal import medfilt

spike_idx = 1000
channel_data = channel_data.copy()
channel_data[spike_idx] += 200.0

windows = [5, 11, 21]
filtered = {}
for w in windows:
    filtered[w] = medfilt(channel_data, kernel_size=w)
print(f'Injected spike at sample {spike_idx} (+200 uV)')
print(f'Applied median filter with windows: {windows}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- الشوكةُ الحمراءُ تَظهرُ في الإشارةِ الأصليةِ (أعلى)
- المُرشِّحُ الوسيطُ يُزيلُ الشوكةَ تمامًا في كلِّ النوافذِ
- النوافذُ الأكبرُ تُنعمُ الإشارةَ أكثرَ لكنّها قد تُخفي تفاصيلَ حقيقيةً
- قارنْ هذا بالمُرشِّحاتِ الأخرى التي تَنشرُ الشوكةَ بدلَ إزالتِها

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=('Original with spike (P4)',
                                    'Median (window=5)',
                                    'Median (window=11)',
                                    'Median (window=21)'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Raw',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=[t_sec[spike_idx]], y=[channel_data[spike_idx]],
                         mode='markers', marker=dict(color='red', size=6),
                         name='Spike'), row=1, col=1)
for i, w in enumerate(windows, start=2):
    fig.add_trace(go.Scatter(x=t_sec, y=filtered[w][:n_plot],
                             name=f'w={w}', line=dict(width=0.5)), row=i, col=1)
fig.update_layout(height=900,
                  title_text='Median Filter - Channel P4 with Artificial Spike',
                  xaxis4_title='Time (s)', showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- يَستبدلُ المُرشِّحُ الوسيطُ كلَّ عيّنةٍ بوسيطِ جيرانِها
- يُقاومُ الشوائبَ والقيمَ الشاذّةَ بشكلٍ فعّالٍ
- لا يَنشرُ الشوكةَ كما يَفعلُ المتوسطُ المتحرّكُ والغاوسيُّ
- يُناسبُ إزالةَ الشوائبِ قبلَ تطبيقِ مُرشِّحاتٍ أخرى

</div>